In [ ]:
import os
import math
from dotenv import load_dotenv
import requests

load_dotenv()

In [ ]:
API_TOKEN = os.getenv("TODOIST_API_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {API_TOKEN}",
    "Content-Type": "application/json"
}

BASE_URL = "https://api.todoist.com/api/v1"
TEMPLATE_PROJECT_NAME = "Packing list"

# Trip Details

In [ ]:
trip_name = input(
    "══════════════════════════════════════════\n"
    "          PACKING LIST GENERATOR\n"
    "══════════════════════════════════════════\n\n"
    "Trip name: "
).strip()

days = int(input("\nHow many days is the trip? ").strip())

season = ""
while season not in ("summer", "winter"):
    season = input("\nSeason — type 'summer' or 'winter': ").strip().lower()

activities_raw = input(
    "\nActivities — enter all that apply, comma-separated.\n"
    "  Options: fancy | professional | dancing | swimming | hiking | skiing | running\n"
    "  Example: dancing, swimming, fancy\n"
    "> "
)
activities = [a.strip().lower() for a in activities_raw.split(",") if a.strip()]

dance_occasions = 0
if "dancing" in activities:
    dance_occasions = int(input("\nHow many dance occasions will you have? ").strip())

professional_occasions = 0
if "professional" in activities:
    professional_occasions = int(input("\nHow many professional occasions will you have? ").strip())

print(f"\n✓ {trip_name} | {days} days | {season}")
print(f"  Activities: {', '.join(activities) if activities else 'none'}")
if dance_occasions:
    print(f"  Dance occasions: {dance_occasions}")
if professional_occasions:
    print(f"  Professional occasions: {professional_occasions}")

# Clothing Formula

In [ ]:
def calculate_clothes(days, season, activities, dance_occasions, professional_occasions):
    tasks = []

    # Base clothing quantities
    tasks.append(f"Panties x{days + 1}")
    tasks.append(f"Socks x{days + 1}")
    tasks.append(f"Bras x{max(1, math.ceil(days / 3))}")
    tasks.append(f"Shirts / tops x{min(days + 1, 10)}")
    tasks.append("Pajamas x1")

    # Bottoms — season dependent
    if season == "summer":
        tasks.append(f"Shorts x{math.ceil(days / 2)}")
        tasks.append(f"Pants / jeans x{max(1, math.ceil(days / 4))}")
        tasks.append(f"Skirts x{max(1, math.ceil(days / 4))}")
        tasks.append(f"Dresses x{max(1, math.ceil(days / 4))}")
        tasks.append("Light cardigan / sweater x1 (for cooler evenings)")
    else:
        tasks.append(f"Pants / jeans x{math.ceil(days / 2)}")
        tasks.append("Warm sweater / fleece x3")
        tasks.append("Winter jacket x1")
        tasks.append("Thermals x1")
        tasks.append("Warm socks x2")
        tasks.append("Hats / scarves x1")
        tasks.append("Tights x2")

    # Activity: fancy
    if "fancy" in activities:
        tasks.append(f"Fancy / formal outfit x{min(2, math.ceil(days / 4))}")

    # Activity: professional
    if "professional" in activities:
        tasks.append(f"Professional outfit x{min(professional_occasions, 3)}")

    # Activity: dancing
    if "dancing" in activities:
        tasks.append(f"Dance outfit x{1 if dance_occasions <= 2 else 2}")

    # Activity: swimming
    if "swimming" in activities:
        tasks.append("Bathing suit x2")
        tasks.append("Goggles x1")
        tasks.append("Waterproof pouch x1")

    # Activity: hiking
    if "hiking" in activities:
        tasks.append("Hiking pants x1")
        tasks.append("Hiking socks x2")

    # Activity: skiing
    if "skiing" in activities:
        tasks.append("Ski suit x1")
        tasks.append("Ski socks x3")
        tasks.append("Thermals x1")
        tasks.append("Balaclava x1")
        tasks.append("Card holder x1")
        tasks.append("Ski card x1")

    # Activity: running
    if "running" in activities:
        tasks.append("Running socks x3")

    # Always
    tasks.append("Purse / day bag x1")
    tasks.append("Laundry bag x1")

    return tasks


def calculate_shoes(season, activities):
    tasks = []

    tasks.append("Comfortable walking shoes x1")

    if season == "summer":
        tasks.append("Flip flops x1")

    if "swimming" in activities:
        tasks.append("Water shoes x1")

    if "fancy" in activities or "professional" in activities:
        tasks.append("Dress shoes x1")

    if "dancing" in activities:
        tasks.append("Dance shoes x1")

    if "hiking" in activities:
        tasks.append("Hiking shoes x1")

    if "skiing" in activities:
        tasks.append("Ski shoes x1")

    if "running" in activities:
        tasks.append("Running shoes x1")

    return tasks


clothes_tasks = calculate_clothes(days, season, activities, dance_occasions, professional_occasions)
shoes_tasks = calculate_shoes(season, activities)

print("Clothes:")
for t in clothes_tasks:
    print(f"  - {t}")
print("\nShoes:")
for t in shoes_tasks:
    print(f"  - {t}")

# API Functions

In [ ]:
def get_projects():
    all_projects = []
    cursor = None
    while True:
        params = {"limit": 50}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(f"{BASE_URL}/projects", headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        all_projects.extend(data["results"])
        cursor = data.get("next_cursor")
        if not cursor:
            break
    return all_projects


def create_project(name):
    resp = requests.post(f"{BASE_URL}/projects", headers=HEADERS, json={"name": name})
    resp.raise_for_status()
    return resp.json()


def get_sections(project_id):
    all_sections = []
    cursor = None
    while True:
        params = {"limit": 50, "project_id": project_id}
        if cursor:
            params["cursor"] = cursor
        resp = requests.get(f"{BASE_URL}/sections", headers=HEADERS, params=params)
        resp.raise_for_status()
        data = resp.json()
        all_sections.extend(data["results"])
        cursor = data.get("next_cursor")
        if not cursor:
            break
    return all_sections


def create_section(name, project_id):
    resp = requests.post(
        f"{BASE_URL}/sections",
        headers=HEADERS,
        json={"name": name, "project_id": project_id}
    )
    resp.raise_for_status()
    return resp.json()


def get_tasks(project_id, section_id=None):
    params = {"project_id": project_id}
    if section_id:
        params["section_id"] = section_id
    resp = requests.get(f"{BASE_URL}/tasks", headers=HEADERS, params=params)
    resp.raise_for_status()
    return resp.json()["results"]


def create_task(content, project_id, section_id=None):
    payload = {"content": content, "project_id": project_id}
    if section_id:
        payload["section_id"] = section_id
    resp = requests.post(f"{BASE_URL}/tasks", headers=HEADERS, json=payload)
    resp.raise_for_status()
    return resp.json()

# Fetch Template

In [ ]:
projects = get_projects()
template = next((p for p in projects if p["name"] == TEMPLATE_PROJECT_NAME), None)
assert template is not None, f"Project '{TEMPLATE_PROJECT_NAME}' not found in Todoist"

template_sections = get_sections(template["id"])

# Build {section_name: [task_content, ...]} map from template
template_tasks = {}
for sec in template_sections:
    template_tasks[sec["name"]] = [
        t["content"] for t in get_tasks(template["id"], sec["id"])
    ]

print(f"Template loaded: {len(template_sections)} sections")
for name, tasks in template_tasks.items():
    print(f"  {name}: {len(tasks)} tasks")

# Create Packing List Project

In [ ]:
project_name = f"{trip_name} — Packing List"
new_project = create_project(project_name)
new_project_id = new_project["id"]

print(f"Created project: '{project_name}' (id={new_project_id})")

# Populate Sections and Tasks

In [ ]:
for sec in template_sections:
    sec_name = sec["name"]
    new_sec = create_section(sec_name, new_project_id)
    new_sec_id = new_sec["id"]

    if sec_name == "Clothes":
        for content in clothes_tasks:
            create_task(content, new_project_id, new_sec_id)
        print(f"  [{sec_name}] {len(clothes_tasks)} tasks (generated)")
    else:
        tasks = template_tasks.get(sec_name, [])
        for content in tasks:
            create_task(content, new_project_id, new_sec_id)
        print(f"  [{sec_name}] {len(tasks)} tasks (copied)")

# Shoes section (not in template — generated separately)
shoes_sec = create_section("Shoes", new_project_id)
for content in shoes_tasks:
    create_task(content, new_project_id, shoes_sec["id"])
print(f"  [Shoes] {len(shoes_tasks)} tasks (generated)")

print(f"\nDone! Open '{project_name}' in Todoist.")